# Processes

- Process = Program in Execution
- Has its own memory (code, data, stack, heap)
- Has unique PID
- Independent from other processes.  They don’t share memory unless you explicitly set up inter‑process communication (IPC) (pipes, shared memory, sockets, etc.).

# fork()
fork() is a system call used in the C programming language on Unix-like systems such as Linux to create a new process.

Basic idea

fork() duplicates the current process.

The new process is called the child.

The original process is the parent.

After fork():


```bash


Parent process
      |
      | fork()
      |
Child process



```

# Example 1 fork() for hello world

In [6]:
%%bash
cat <<EOF > fork_example.c
#include <stdio.h>
#include <unistd.h>

int main() {
    fork();

    // below fork every thing gets done twice because we have a child process now
    printf("Hello World\n");
    return 0;
}




EOF

gcc fork_example.c -o fork_example
./fork_example

Hello World
Hello World


- The .c file = source code

- The compiled executable = program on disk

- When you run ./fork_example = becomes a PROCESS


```bash

Process 1 (Parent) ──┐
                     ├── both run the same printf line
Process 2 (Child)  ──┘

```

# Example 2 fork()s for hello world

In [19]:
%%bash
cat <<EOF > fork_example.c
#include <stdio.h>
#include <unistd.h>

int main() {
    fork();
    fork();

    // below fork every thing gets done twice because we have a child process now
    printf("Hello World\n");
    return 0;
}
EOF

gcc fork_example.c -o fork_example
./fork_example

Hello World
Hello World
Hello World
Hello World


```bash




1st fork--------Parent creates Child 1
2nd fork--------Parent creates Child 2
                Child 1 creates Child 3

      Parent
     /     \
 Child 1   Child 2   (from Parent's 2nd fork)
    |
 Child 3             (from Child 1's 2nd fork)



```

# How many hello world do we get here? 
**`Answer: its 2^n`**

In [7]:
%%bash
cat <<EOF > fork_example.c
#include <stdio.h>
#include <unistd.h>

int main() {
    fork();
    fork();
    fork();
    printf("Hello World\n");
    return 0;
}
EOF

gcc fork_example.c -o fork_example
./fork_example

Hello World
Hello World
Hello World
Hello World
Hello World
Hello World
Hello World
Hello World


# ID vs getpid()
- id (fork return) = tells you if you're parent (child's PID) or child (0)

- getpid() = tells you your own unique process ID



# Child process has id 0

In [8]:
%%bash
cat <<EOF > fork_example.c
#include <stdio.h>
#include <unistd.h>

int main() {
    int id = fork();
    printf("Hello World from ID: %d\n",id);
    return 0;
}
EOF

gcc fork_example.c -o fork_example
./fork_example

Hello World from ID: 48519
Hello World from ID: 0


# getpid()

In [14]:
%%bash

cat <<EOF > fork_example.c
#include <stdio.h>
#include <unistd.h>

int main() {
    int id = fork();
    
    printf("My own PID: %d, fork() returned: %d\n", getpid(), id);
    
    if (id == 0) {
        printf("  → I am the CHILD (PID: %d). My parent's PID is: %d\n", 
               getpid(), getppid());
    } else {
        printf("  → I am the PARENT (PID: %d). I created a child with PID: %d\n", 
               getpid(), id);
        sleep(40);

    }
    
    return 0;
}
EOF

gcc fork_example.c -o fork_example
./fork_example

My own PID: 49179, fork() returned: 0
  → I am the CHILD (PID: 49179). My parent's PID is: 49178
My own PID: 49178, fork() returned: 49179
  → I am the PARENT (PID: 49178). I created a child with PID: 49179


# Lets get three hello worlds
So we make child process for the master process and then child for the master only


In [15]:
%%bash
cat <<EOF > fork_example.c
#include <stdio.h>
#include <unistd.h>

int main() {

   int id =  fork();   // create child process
if (id !=0)
{fork();}   // create child process

    printf("Hello World \n");

    return 0;
}
EOF

gcc fork_example.c -o fork_example
./fork_example

Hello World 
Hello World 
Hello World 


# wait()

wait() is a system call that makes a parent process pause execution until one of its child processes terminates. It's used for synchronization between parent and child processes.

# Key Points
- Parent waits: Only the parent can wait() for its children (not vice versa)

- Zombie prevention: Cleans up terminated child processes

- Synchronization: Ensures parent doesn't continue before child finishes

- Blocking: wait() blocks until a child terminates (unless using WNOHANG)

# Example
Lets print 1 2 3 4 5 using the child process and  6 7 8 9 10 using the parent.
We use wait() to make sure child process executes first and parent process 
waits meanwhile.

# without wait()

In [16]:
%%bash
cat <<'EOF' > wait_example.c
#include <stdio.h>
#include <unistd.h>
#include <stdlib.h>

int main(int argc, char* argv[]) {
    int id = fork();
    int n;

    if (id == 0) {
        n = 1;               // child
    } else {
        n = 6;               // parent
    }

    // No wait – both processes race to print
    for (int i = n; i < n + 5; i++) {
        printf("%d ", i);
        fflush(stdout);      // force immediate output
        usleep(10000);  // sleep 10 ms – yields the CPU

    }

    if (id != 0) {
        printf("\n");        // parent prints a newline at the end
    }

    return 0;
}
EOF

gcc wait_example.c -o wait_example
./wait_example

6 1 7 2 8 3 9 4 10 5 


# With wait()

In [17]:
%%bash
cat <<'EOF' > wait_example.c
#include <stdio.h>
#include <unistd.h>
#include <stdlib.h>
#include <sys/wait.h>

int main(int argc, char* argv[]) {
    int id = fork();
    int n;

    if (id == 0) {
        n = 1;
    } else {
        n = 6;
    }

    if (id != 0) {
    // wait till the child process dies
    // parent process goes to sleep here
    // This is the synchronization point that guarantees ordered output.


        wait(NULL); 
    }

    for (int i = n; i < n + 5; i++) {
        printf("%d ", i);
        
        //  fflush() Forces output IMMEDIATELY instead of waiting for buffer to fill. 
        //  Without this, you might see nothing until loop ends or newline appears.
      
      /*
      With multiple processes competing for output, fflush(stdout)
      ensures that when a process prints a number, it immediately appears
      on screen. Without it:
      - The child might print 1 2 3 4 5 into its buffer
      - The parent might print 6 7 8 9 10 into its buffer
      - The actual order you see depends on whose buffer gets flushed first
      - With fflush(), you guarantee that each number appears at the exact 
        moment printf() is called, giving you true interleaving rather 
        than chunk-by-chunk output.
      */
    
        fflush(stdout);
        usleep(10000);  // sleep 10 ms – yields the CPU

   
}

    if (id != 0) {
        printf("\n");
    }

 
 return 0;

}
EOF

gcc wait_example.c -o wait_example
./wait_example

1 2 3 4 5 6 7 8 9 10 


In [54]:
"""
fork() return = "Who am I to my parent?"
getpid()      = "What's my government ID number?"

"""

'\nfork() return = "Who am I to my parent?"\ngetpid()      = "What\'s my government ID number?"\n\n'

# getppid()

In [18]:
%%bash
cat <<'EOF' > wait_example.c
#include <stdio.h>
#include <unistd.h>
#include <stdlib.h>
#include <sys/wait.h>

int main(int argc, char* argv[]) {
    int id = fork();
    
    // Print BOTH: the fork() return value AND your own PID
    printf("Current ID: %d, Parent ID: %d\n", getpid(), getppid());
    
    return 0;
}
EOF

gcc wait_example.c -o wait_example

./wait_example

Current ID: 50414, Parent ID: 50407
Current ID: 50415, Parent ID: 50414


```bash


[TERMINAL/bash] (PID: 50407)  <── THIS IS THE "PARENT OF THE PARENT"
    │
    ├── [YOUR PROGRAM] (PID: 50414)  <── PARENT process
    │    │
    │    └── [CHILD PROCESS] (PID: 50415)  <── CHILD
    │
    └── [OTHER PROGRAMS YOU RUN] (other PIDs)
         (ls, gcc, etc.)

```